In [13]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [14]:
load_dotenv()

False

In [ ]:
import os
from langchain_groq import ChatGroq

GROQ_API_KEY
GROQ_API_KEY


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [16]:
class JokeState(TypedDict):
    joke: str
    topic: str
    explaination: str

In [20]:
def generate_joke(state: JokeState) -> JokeState:
    prompt= f'Generate a joke about {state["topic"]}.'
    response = llm.invoke(prompt).content

    return {'joke': response}

def explain_joke(state: JokeState) -> JokeState:
    
    prompt = f'Explain the joke on the topic {state["topic"]} and joke {state["joke"]}.'
    response = llm.invoke(prompt).content
    
    return{'explanation': response}

In [21]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('explain_joke', explain_joke)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer = checkpointer)

In [22]:
config1 = {'configurable':{'thread_id': '1'}}
workflow.invoke({'topic':'pizza'},config = config1)

{'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.',
 'topic': 'pizza'}

In [23]:
workflow.get_state(config1)

StateSnapshot(values={'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.', 'topic': 'pizza'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15902e-5f13-6a91-8002-3243ce2cb64a'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-26T13:00:40.779432+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15902e-549a-6bf3-8001-9f9d2a815d96'}}, tasks=(), interrupts=())

In [24]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.', 'topic': 'pizza'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15902e-5f13-6a91-8002-3243ce2cb64a'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-26T13:00:40.779432+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15902e-549a-6bf3-8001-9f9d2a815d96'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.', 'topic': 'pizza'}, next=('explain_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15902e-549a-6bf3-8001-9f9d2a815d96'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-05-26T13:00:39.681329+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15902e-504a-6315-8

In [25]:
config2 = {'configurable':{'thread_id':'2'}}
workflow.invoke({'topic':'Narendra Mode'}, config = config2)

{'joke': "I think you meant Narendra Modi. Here's a joke:\n\nWhy did Narendra Modi bring a ladder to the meeting?\n\nBecause he wanted to take his policies to the next level.",
 'topic': 'Narendra Mode'}

In [26]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'joke': "I think you meant Narendra Modi. Here's a joke:\n\nWhy did Narendra Modi bring a ladder to the meeting?\n\nBecause he wanted to take his policies to the next level.", 'topic': 'Narendra Mode'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f159037-e6d5-6e1c-8002-5bada52f320f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-26T13:04:56.606671+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f159037-e027-6d83-8001-2eef48e43867'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'joke': "I think you meant Narendra Modi. Here's a joke:\n\nWhy did Narendra Modi bring a ladder to the meeting?\n\nBecause he wanted to take his policies to the next level.", 'topic': 'Narendra Mode'}, next=('explain_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f159037-e027-6d83-8001-2eef48e43867'}}, metadata={'

Time Travel